# Quickstart: ontology → graph → query → reasoning → validation → metrics

One notebook, the whole repo in miniature, on the retail example. Everything runs locally (no database, no cloud account).

```bash
pip install -r requirements.txt   # from the repo root
jupyter lab notebooks/quickstart.ipynb
```

In [1]:
import sys
from pathlib import Path

from rdflib import Graph, Namespace

RETAIL = Path.cwd().parent / "examples/retail"
sys.path.insert(0, str(RETAIL / "scripts"))
R = Namespace("https://example.org/retail/ontology#")
EX = Namespace("https://example.org/retail/data/")

ontology = Graph().parse(RETAIL / "ontology/retail-ontology.ttl")   # the schema (TBox)
data = Graph().parse(RETAIL / "rdf/retail-data.ttl")                # the data (ABox)
kg = ontology + data                                                # schema + data = knowledge graph
print(f"ontology: {len(ontology)} triples · data: {len(data)} triples · knowledge graph: {len(kg)} triples")

ontology: 238 triples · data: 165 triples · knowledge graph: 403 triples


## 1. The schema is data too

In RDF the ontology is just more triples, so you can query the model itself.

In [2]:
q = '''
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?class ?super WHERE {
  ?class a owl:Class . FILTER(isIRI(?class))
  OPTIONAL { ?class rdfs:subClassOf ?super . FILTER(isIRI(?super)) }
} ORDER BY ?class'''
for row in kg.query(q):
    print(row["class"].n3(kg.namespace_manager), "⊑", row["super"].n3(kg.namespace_manager) if row["super"] else "—")

retail:Address ⊑ schema:PostalAddress
retail:Category ⊑ —
retail:Customer ⊑ schema:Person
retail:Customer ⊑ retail:Party
retail:Order ⊑ schema:Order
retail:OrderLine ⊑ —
retail:Party ⊑ —
retail:PremiumCustomer ⊑ retail:Customer
retail:Product ⊑ —
retail:Review ⊑ schema:Review
retail:Seller ⊑ schema:Organization
retail:Seller ⊑ retail:Party


## 2. Query the data with SPARQL

Revenue per customer = Σ quantity × unit price, over every order line (`sparql/03-order-totals-per-customer.rq`).

In [3]:
q = (RETAIL / "sparql/03-order-totals-per-customer.rq").read_text()
for row in data.query(q):
    print(f"{row.customerName:15} orders={row.orders}  revenue={row.totalSpent}")

Alice Johnson   orders=2  revenue=1498.97
Bob Smith       orders=1  revenue=199.98
Carol Diaz      orders=1  revenue=179.48


## 3. Reasoning: facts nobody wrote down

Alice and Carol are typed only as `PremiumCustomer`. Is Alice a `Customer`? Not until a reasoner applies the ontology.

In [4]:
import owlrl
from rdflib.namespace import RDF

print("before reasoning:", (EX.C1, RDF.type, R.Customer) in kg)
reasoned = Graph()
for t in kg:
    reasoned.add(t)
reasoned.parse(RETAIL / "ontology/retail-defined-classes.ttl")
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(reasoned)
print("after reasoning: ", (EX.C1, RDF.type, R.Customer) in reasoned)
print("Alice purchased (property chain):", sorted(p.split('/')[-1] for p in reasoned.objects(EX.C1, R.purchased)))
print("LaptopBuyer (defined class):     ", [s.split('/')[-1] for s in reasoned.subjects(RDF.type, R.LaptopBuyer)])

before reasoning: False


after reasoning:  True
Alice purchased (property chain): ['P1', 'P2', 'P3']
LaptopBuyer (defined class):      ['C1']


## 4. Validation: reject bad data with SHACL

OWL infers; SHACL checks. Break a review (7 stars) and see it rejected.

In [5]:
from pyshacl import validate
from rdflib import Literal

shapes = Graph().parse(RETAIL / "ontology/retail-shapes.ttl")
ok, _, _ = validate(data, shacl_graph=shapes, ont_graph=ontology, inference="rdfs")
print("real data conforms:", ok)

broken = Graph()
for t in data:
    broken.add(t)
broken.remove((EX.R1, R.rating, None))
broken.add((EX.R1, R.rating, Literal(7)))
ok, _, report = validate(broken, shacl_graph=shapes, ont_graph=ontology, inference="rdfs")
print("broken data conforms:", ok)
print([line.strip() for line in report.splitlines() if "Message" in line][0])

real data conforms: True


broken data conforms: False
Message: Rating must be an integer between 1 and 5.


## 5. The same data as a property graph

The Neptune openCypher CSVs (generated from the RDF) answer \"customers who bought the earbuds also bought…\" with a 4-hop walk.

In [6]:
import check_consistency as cc

nodes, edges = cc.load_pg(cc.AWS / "opencypher-csv", ":ID", ":LABEL", ":TYPE", ":START_ID", ":END_ID")
print(f"{len(nodes)} nodes, {len(edges)} edges")
q2, q3, q5, q7 = cc.pg_answers(nodes, edges)
print("also bought:", q5)
print("never ordered:", q7)

29 nodes, 41 edges
also bought: {'Laptop Pro 14': 1, 'USB-C Charger 65W': 1, 'Knowledge Graphs 101': 1}
never ordered: ['Mechanical Keyboard']


## 6. The same data as a semantic model

Load the warehouse tables into SQLite and compute the Apache Ossie metrics: the same numbers as SPARQL above.

In [7]:
import check_semantic_model as cs
import yaml

db = cs.load_warehouse()
model = yaml.safe_load((RETAIL / "semantic-model/ossie/retail.semantic_model.yaml").read_text())
revenue = next(m for d in model["datasets"] for m in d.get("metrics", []) if m["name"] == "revenue")
sql = cs.compile_metric(model, revenue, "order_lines")
print("Ossie metric 'revenue' compiles to:\n ", sql)
print("total revenue:", round(db.execute(sql).fetchone()[0], 2))

Ossie metric 'revenue' compiles to:
  SELECT SUM((order_lines.quantity * order_lines.unit_price)) FROM order_lines AS order_lines
total revenue: 1878.43


## Where next

* Concepts: [ontologies](../ontologies/README.md) · [graphs](../graphs/README.md) · [semantic models](../semantic-models/README.md) · [foundations](../foundations/README.md)
* Products: the [provider index](../README.md#provider-index)
* Try editing `examples/retail/rdf/retail-data.ttl`, then run `make generate && make check`